# Lesson 3.2 — experiments: alpha-beta pruning

Run the cells in order. Before each experiment, write down your prediction in the "Your prediction" cell (double-click it to edit), and only then run the code.

The first cell chooses the implementation:

- `"mine"` uses your `task.py`; solve Part B of the task first;
- `"reference"` uses the course's reference implementation. Set `IMPLEMENTATION = "reference"` if you want to run the notebook before finishing the task.

In [ ]:
%load_ext autoreload
%autoreload 2
import os

# "mine" uses your task.py; "reference" uses the course's reference implementation.
IMPLEMENTATION = os.environ.get("INTROAI_IMPLEMENTATION", "mine")
if IMPLEMENTATION == "mine":
    import task as algorithms
else:
    from introai.games import reference as algorithms

The function `run` runs one search and returns its result, the number of positions visited and the time in seconds, as in lesson 3.1. The function `ordered(cells)` creates a tic-tac-toe in which `game.actions` lists the empty cells in the order of the list `cells`. The rules of the game do not change; only the order in which the search looks at the moves does. Minimax and alpha-beta with strict inequalities come from the course's reference version.

In [ ]:
import random
import time
from statistics import mean

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

from introai.games import TIC_TAC_TOE, KInARow, reference
from introai.testing.games import positions_visited
from introai.trace import TraceLimitExceeded, record

HINT = 'check task.py (or set IMPLEMENTATION = "reference" in the first cell)'
CELLS = [(row, col) for row in range(3) for col in range(3)]


def run(search, game, state, limit=1_000_000):
    """Run one search; return its result, the positions visited and the time in seconds."""
    start = time.perf_counter()
    try:
        with record(limit=limit, kinds={"result"}) as trace:
            result = search(game, state)
    except NotImplementedError:
        raise RuntimeError(f"{search.__name__} is not written yet - {HINT}") from None
    except (TraceLimitExceeded, RecursionError):
        message = f"{search.__name__} visited more than {limit} positions - {HINT}"
        raise RuntimeError(message) from None
    return result, positions_visited(trace), time.perf_counter() - start


class Ordered(KInARow):
    """Tic-tac-toe whose moves come in a fixed order of the cells."""

    def __init__(self, cells):
        super().__init__(3, 3, 3)
        self.rank = {cell: i for i, cell in enumerate(cells)}

    def actions(self, state):
        return sorted(super().actions(state), key=self.rank.__getitem__)


def ordered(cells):
    return Ordered(cells)


def strict_alpha_beta(game, state):
    """The reference alpha-beta that cuts off only at v > beta and v < alpha."""
    return reference.alpha_beta_search(game, state, strict=True)


def empty_cells(state):
    return sum(row.count(".") for row in state)


def random_position(empty, seed):
    """A position with `empty` empty cells after random moves, or None if the game ended."""
    rng = random.Random(seed)
    state = TIC_TAC_TOE.initial
    while empty_cells(state) > empty and not TIC_TAC_TOE.is_terminal(state):
        state = TIC_TAC_TOE.result(state, rng.choice(TIC_TAC_TOE.actions(state)))
    return None if TIC_TAC_TOE.is_terminal(state) else state

## Experiment 1: how many positions from the empty board

Four searches look for the first move on the empty board: minimax, your alpha-beta, your alpha-beta on a board where it looks at the centre first, then the corners and the sides last, and the reference alpha-beta with strict inequalities.

**Your prediction.** How many times fewer positions than minimax will alpha-beta visit? Which order of the cells will be better? What do the strict inequalities cost?

*My prediction:* …

In [ ]:
centre_first = ordered([(1, 1), (0, 0), (0, 2), (2, 0), (2, 2), (0, 1), (1, 0), (1, 2), (2, 1)])
searches = [
    ("minimax (reference)", reference.minimax_search, TIC_TAC_TOE),
    ("alpha-beta", algorithms.alpha_beta_search, TIC_TAC_TOE),
    ("alpha-beta, centre and corners first", algorithms.alpha_beta_search, centre_first),
    ("alpha-beta, strict cutoffs (reference)", strict_alpha_beta, TIC_TAC_TOE),
]
visited = {}
print(f"{'':<40}{'value':>6}{'positions':>11}{'seconds':>9}")
for label, search, game in searches:
    (value, _), visited[label], seconds = run(search, game, game.initial)
    print(f"{label:<40}{value:>6}{visited[label]:>11}{seconds:>9.2f}")

**Think about it.** Did all four searches give the same value? Why does the order "centre and corners first" help? Why are strict inequalities so costly in tic-tac-toe?

## Experiment 2: 200 random orders of the cells

For each of 200 seeds we shuffle the nine cells and count how many positions alpha-beta visits from the empty board. The histogram shows how often each number of positions comes up. The dashed lines are the natural order (row by row) and "centre and corners first". The computation takes about half a minute.

**Your prediction.** How many positions will alpha-beta visit with the best and with the worst of the random orders? Where on the histogram will the natural order fall?

*My prediction:* …

In [ ]:
counts = []
for seed in range(200):
    cells = CELLS[:]
    random.Random(seed).shuffle(cells)
    game = ordered(cells)
    _, positions, _ = run(algorithms.alpha_beta_search, game, game.initial)
    counts.append(positions)
print(f"fewest {min(counts)}, most {max(counts)}, mean {mean(counts):.0f}")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(counts, bins=30, color="#2a78d6", edgecolor="#fcfcfb", linewidth=1)
markers = [
    ("natural order", visited["alpha-beta"], "#eb6834"),
    ("centre and corners first", visited["alpha-beta, centre and corners first"], "#1baf7a"),
]
for label, x, color in markers:
    ax.axvline(x, color=color, linewidth=2, linestyle="--", label=f"{label}: {x}")
ax.set_title(
    "Positions alpha-beta visits from the empty board, 200 random cell orders",
    loc="left",
    color="#0b0b0b",
)
ax.set_xlabel("positions visited", color="#52514e")
ax.set_ylabel("cell orders", color="#52514e")
ax.yaxis.set_major_locator(MaxNLocator(integer=True))
ax.legend(frameon=False, loc="upper right", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** How many times more positions does the worst random order visit than the best one? Did any random order beat "centre and corners first"? What does that say about the importance of move order in games whose trees are far larger?

## Experiment 3: savings and the number of empty cells

For every number of empty cells from 2 to 8 we draw 10 positions (seeded, by random moves) and count how many positions minimax and your alpha-beta visit from them on average. With 9 empty cells there is only the empty board of experiment 1. The dashed line is the square root of minimax's count, what you can expect with a perfect move order. The vertical axis has a logarithmic scale.

**Your prediction.** With how many empty cells does alpha-beta stop saving anything? How do the savings change as the number of empty cells grows?

*My prediction:* …

In [ ]:
sizes = list(range(2, 10))
minimax_mean, alpha_beta_mean = [], []
for empty in sizes:
    if empty == 9:
        minimax_mean.append(visited["minimax (reference)"])
        alpha_beta_mean.append(visited["alpha-beta"])
        continue
    positions, seed = [], 0
    while len(positions) < 10:
        state = random_position(empty, 1000 * empty + seed)
        seed += 1
        if state is not None and empty_cells(state) == empty:
            positions.append(state)
    minimax_mean.append(mean(run(reference.minimax_search, TIC_TAC_TOE, s)[1] for s in positions))
    alpha_beta_mean.append(
        mean(run(algorithms.alpha_beta_search, TIC_TAC_TOE, s)[1] for s in positions)
    )
print(f"{'empty cells':<13}{'minimax':>10}{'alpha-beta':>12}{'ratio':>8}")
for empty, m, a in zip(sizes, minimax_mean, alpha_beta_mean):
    print(f"{empty:<13}{m:>10.0f}{a:>12.0f}{a / m:>8.3f}")

fig, ax = plt.subplots(figsize=(9, 4))
for label, values, color in (
    ("minimax", minimax_mean, "#2a78d6"),
    ("alpha-beta", alpha_beta_mean, "#eb6834"),
):
    ax.plot(
        sizes,
        values,
        color=color,
        linewidth=2,
        marker="o",
        markersize=8,
        markeredgecolor="#fcfcfb",
        markeredgewidth=1.5,
        label=label,
    )
ax.plot(
    sizes,
    [m**0.5 for m in minimax_mean],
    color="#52514e",
    linewidth=2,
    linestyle="--",
    label="square root of minimax (perfect order)",
)
ax.set_yscale("log")
ax.set_xticks(sizes)
ax.set_title(
    "Positions visited, by the number of empty cells (log scale)", loc="left", color="#0b0b0b"
)
ax.set_xlabel("empty cells", color="#52514e")
ax.legend(frameon=False, loc="upper left", labelcolor="#0b0b0b")
ax.grid(axis="y", color="#e1e0d9", linewidth=0.8)
ax.set_axisbelow(True)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
for side in ("left", "bottom"):
    ax.spines[side].set_color("#c3c2b7")
ax.tick_params(colors="#52514e")
fig.tight_layout()

**Think about it.** Why does alpha-beta have nothing to prune with two empty cells? How fast does the advantage of alpha-beta grow with every extra empty cell? Does your alpha-beta come close to the dashed line or rather to minimax?